# 07 — Open-source LLMs for SOAP generation

Answers the supervisor's question with a measurement: **is an open-source LLM
better than the extractive Bio_ClinicalBERT pipeline for SOAP notes?**

**Runtime > Change runtime type > T4 GPU** before running anything.

## What is being compared

Every model gets *byte-identical input*: the sentences the extractive
classifier already selected for each section, frozen in `llm_handoff.json`.
The model rewrites them as prose. It never sees the raw transcript, so it
cannot import a fact from elsewhere in the conversation — that is the hybrid
contract, and it is what bounds hallucination structurally rather than by
asking nicely.

## What is being measured

| | What it means | Extractive baseline |
|---|---|---|
| Clinical accuracy | did each reference sentence reach the right section | **97.4%** |
| Novel content | words in the note with no source | **0.0%** |
| Omission | source content the note dropped | **0.0%** |
| Numerics not in source | invented or altered doses | **0** |

The last one is not a quality metric. An invented dose is a patient-safety
defect, so it is reported separately and never averaged into anything.

Extraction scores perfectly on the last three *by construction* — every word
comes from the transcript. The question this notebook asks is what a model's
fluency is worth against that.

## Precedent worth remembering

BioGPT was tried on this project and removed: it ignored the instruction and
performed autoregressive completion, echoing the doctor's greeting as the
Subjective section. **Meditron 7B is a base model too** — continued pretraining,
no instruction tuning. If it fails the same way, that is a finding, not a
surprise.

In [ ]:
# ---- 1. environment -------------------------------------------------------
# The repo is cloned for three things: the ground truth, the scoring functions,
# and the PRODUCTION PROMPT. Copying the prompt into this notebook would let it
# drift from what the backend actually sends, and then the measurement would
# describe a system that does not exist.

!git clone -q https://github.com/Wajeeha-Kamran/emr-assistant-backend.git
%cd emr-assistant-backend

!pip -q install "transformers>=4.44" accelerate bitsandbytes sentencepiece

import torch, sys
sys.path.insert(0, "/content/emr-assistant-backend")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to T4")

In [ ]:
# ---- 2. the frozen input --------------------------------------------------
import json, time, gc, csv, os

HANDOFF = "docs/evidence/benchmarks/llm_handoff.json"
data = json.load(open(HANDOFF, encoding="utf-8"))

SELECTIONS = data["selections"]      # {script: {section: [sentence, ...]}}
EXTRACTIVE = data["extractive"]      # the control, same selections rendered verbatim
EXPECTED   = data["expected"]        # [{label, sentence}, ...] per script

for s in sorted(SELECTIONS, key=int):
    print(f"script {s}: "
          f"{ {k: len(v) for k, v in SELECTIONS[s].items()} }  "
          f"{len(EXPECTED[s])} labelled sentences")

In [ ]:
# ---- 3. scoring: imported, not reimplemented ------------------------------
# Same functions the backend's own evaluation uses. Neither module pulls in
# torch or ClinicalBERT at import time, so this is cheap.

from scripts.evaluate_soap import SECTION_OF_LABEL, words, contains
from scripts.evaluate_soap_e2e import sections_containing, COVERAGE_THRESHOLD
from scripts.evaluate_groundedness import score_section

SECTIONS = ("subjective", "objective", "assessment", "plan")
print("coverage threshold:", COVERAGE_THRESHOLD)


def score_note(script, note):
    """One consultation -> section accuracy + groundedness."""
    section_words = {k: words(v) for k, v in note.items()}
    right = total = leaked = noise = 0

    for item in EXPECTED[script]:
        label, sentence = item["label"], item["sentence"]
        found = sections_containing(words(sentence), section_words)
        if label == "X":
            noise += 1
            leaked += 1 if found else 0
            continue
        total += 1
        right += 1 if SECTION_OF_LABEL[label] in found else 0

    novel = note_n = om = src_n = 0
    bad = []
    for name in SECTIONS:
        a, b, c, d, _, e = score_section(SELECTIONS[script].get(name) or [],
                                         note.get(name, ""))
        novel += a; note_n += b; om += c; src_n += d; bad += e

    return {
        "clinical_right": right, "clinical_total": total,
        "noise_leaked": leaked, "noise_total": noise,
        "novel": novel, "novel_total": note_n,
        "omitted": om, "omitted_total": src_n,
        "bad_numerics": bad,
    }


def summarise(label, notes, seconds=None):
    agg = {k: 0 for k in ("clinical_right", "clinical_total", "noise_leaked",
                          "noise_total", "novel", "novel_total",
                          "omitted", "omitted_total")}
    all_bad = []
    for s in sorted(SELECTIONS, key=int):
        r = score_note(s, notes[s])
        for k in agg:
            agg[k] += r[k]
        all_bad += r["bad_numerics"]

    pct = lambda a, b: (a / b * 100) if b else 0.0
    row = {
        "model": label,
        "clinical_acc": round(pct(agg["clinical_right"], agg["clinical_total"]), 1),
        "noise_rate": round(pct(agg["noise_leaked"], agg["noise_total"]), 1),
        "novel_rate": round(pct(agg["novel"], agg["novel_total"]), 1),
        "omission_rate": round(pct(agg["omitted"], agg["omitted_total"]), 1),
        "bad_numerics": len(all_bad),
        "seconds_per_note": round(seconds, 1) if seconds else "",
    }
    print(f"\n{label}")
    print(f"  clinical accuracy   {row['clinical_acc']}%   (extractive 97.4%)")
    print(f"  noise rate          {row['noise_rate']}%   (extractive 0.0%)")
    print(f"  novel content       {row['novel_rate']}%   (extractive 0.0%)")
    print(f"  omission            {row['omission_rate']}%   (extractive 0.0%)")
    print(f"  bad numerics        {row['bad_numerics']}"
          f"{'   <-- SAFETY DEFECT' if all_bad else ''}")
    if all_bad:
        print(f"    {sorted(set(all_bad))}")
    return row


RESULTS = [summarise("extractive (control)", EXTRACTIVE)]

In [ ]:
# ---- 4. generation --------------------------------------------------------
# The prompt is imported from the backend so this measures what the backend
# would actually send. do_sample=False makes runs reproducible; with sampling
# on, two runs of the same model give two different answers and the comparison
# stops being one.

from app.ml.llm_soap_engine import PROMPT, SECTION_BRIEF, LLMSoapEngine
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

FALLBACK_TEXT = "Not documented in dialogue."


def load(model_id, four_bit=True):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map="auto", dtype=torch.float16)
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.eval()
    return tok, model


def generate_notes(model_id, four_bit=True, max_new_tokens=220):
    tok, model = load(model_id, four_bit)
    notes, t0 = {}, time.time()

    for script in sorted(SELECTIONS, key=int):
        note = {}
        for name in SECTIONS:
            picked = [s.strip() for s in (SELECTIONS[script].get(name) or []) if s.strip()]
            if not picked:
                note[name] = FALLBACK_TEXT
                continue
            prompt = PROMPT.format(
                section=name.capitalize(), brief=SECTION_BRIEF[name],
                bullets="\n".join(f"- {s}" for s in picked),
            )
            text = (tok.apply_chat_template([{"role": "user", "content": prompt}],
                                            tokenize=False, add_generation_prompt=True)
                    if getattr(tok, "chat_template", None) else prompt)
            inputs = tok(text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False,
                                     pad_token_id=tok.eos_token_id)
            answer = tok.decode(out[0][inputs["input_ids"].shape[-1]:],
                                skip_special_tokens=True)
            note[name] = LLMSoapEngine._clean(answer)
        notes[script] = note
        print(f"  script {script} done")

    per_note = (time.time() - t0) / len(SELECTIONS)
    del model; gc.collect(); torch.cuda.empty_cache()
    return notes, per_note

## Run one model per cell

One at a time, freeing the GPU between. Read the notes before trusting the
numbers — a model can score well and still be obviously wrong in a way no
metric was built to catch, which is how BioGPT was caught.

In [ ]:
# ---- MedGemma 4B ----------------------------------------------------------
# First because it is smallest: the only candidate with a plausible future on
# the CPU-only deployment target. If it is good enough, the hardware question
# goes away.
MODEL = "google/medgemma-4b-it"

notes, secs = generate_notes(MODEL)
RESULTS.append(summarise(MODEL, notes, secs))
json.dump(notes, open("notes_medgemma4b.json", "w"), indent=2)

for k, v in notes["1"].items():
    print(f"\n--- {k} ---\n{v}")

In [ ]:
# ---- MediNote-7B ----------------------------------------------------------
# Purpose-built for clinical notes from dialogue -- the closest task match on
# the supervisor's list. Check the exact repo id on huggingface.co before
# running; it has moved.
MODEL = "AGBonnet/medinote-7b"

notes, secs = generate_notes(MODEL)
RESULTS.append(summarise(MODEL, notes, secs))
json.dump(notes, open("notes_medinote7b.json", "w"), indent=2)

for k, v in notes["1"].items():
    print(f"\n--- {k} ---\n{v}")

In [ ]:
# ---- Llama 3 8B Instruct --------------------------------------------------
# THE CONTROL, and not filler. If a general-purpose instruction-tuned model
# matches or beats the medical ones, the finding is that instruction tuning
# matters more than medical pretraining for this task -- which is what BioGPT's
# failure already suggested. That is a more interesting result for the report
# than "we used the medical one".
#
# Gated: accept the licence on huggingface.co, then huggingface-cli login.
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

notes, secs = generate_notes(MODEL)
RESULTS.append(summarise(MODEL, notes, secs))
json.dump(notes, open("notes_llama3_8b.json", "w"), indent=2)

for k, v in notes["1"].items():
    print(f"\n--- {k} ---\n{v}")

In [ ]:
# ---- Mistral 7B Instruct --------------------------------------------------
MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

notes, secs = generate_notes(MODEL)
RESULTS.append(summarise(MODEL, notes, secs))
json.dump(notes, open("notes_mistral7b.json", "w"), indent=2)

for k, v in notes["1"].items():
    print(f"\n--- {k} ---\n{v}")

In [ ]:
# ---- Meditron 7B (optional, expected to fail) -----------------------------
# epfl-llm/meditron-7b is a BASE model: continued pretraining on medical text,
# no instruction tuning. It is on the supervisor's list, and it is the same
# shape as BioGPT, which this project already removed for ignoring its
# instruction and completing text instead.
#
# Run it to demonstrate that instruction tuning is the property that matters,
# not to adopt it. If it produces something unrelated to the input, that is the
# expected result and it belongs in the report.
MODEL = "epfl-llm/meditron-7b"

notes, secs = generate_notes(MODEL)
RESULTS.append(summarise(MODEL, notes, secs))
json.dump(notes, open("notes_meditron7b.json", "w"), indent=2)

for k, v in notes["1"].items():
    print(f"\n--- {k} ---\n{v}")

In [ ]:
# ---- 5. the comparison ----------------------------------------------------
import pandas as pd

df = pd.DataFrame(RESULTS)
df.to_csv("llm_soap_comparison.csv", index=False)
display(df)

print("""
How to read this:

  A model only earns its place if it holds clinical accuracy near 97.4% AND
  keeps bad_numerics at 0. Novel content above zero is expected and is the
  prose you are paying for -- it is a magnitude, not a verdict.

  A high omission rate means the model is dropping clinical content, which no
  amount of fluency compensates for.

  Any non-zero bad_numerics disqualifies a model for clinical use regardless of
  every other column. Read the flagged values; the metric is deliberately
  biased against false alarms, so anything it does flag is worth checking by
  hand.

Download llm_soap_comparison.csv and the notes_*.json files before the runtime
disconnects.
""")